In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import nltk
# Download NLTK data once (stopwords, wordnet for lemmatization)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --- Data paths (relative to notebook in model/) ---
DATA_DIR = Path("../10_categories_of_yahoo_answers_for_nlp_tasks_csv")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"



## 1. Load data and text preprocessing

- Combine **question_title** and **question_content** into a single text field.
- **Normalize**: lowercase, collapse whitespace, strip.
- **Token-level normalization** (used in TF-IDF): **stopword removal**, **lemmatization** (or optional stemming).
- Target: **class_index** (1–10).

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize_for_vectorizer(text):
    text = text.lower().strip()
    tokens = re.findall(r"\b\w+\b", text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return tokens

def preprocess_df(df):
    title = df["question_title"].fillna("").astype(str)
    content = df["question_content"].fillna("").astype(str)
    combined = (title + " " + content).str.strip()
    return combined.apply(normalize_text)


# Load train (first 50k) and test (first 5k)
train_df = pd.read_csv(TRAIN_PATH, nrows=50_000)
test_df = pd.read_csv(TEST_PATH, nrows=5_000)

# Preprocess text
train_df["text"] = preprocess_df(train_df)
test_df["text"] = preprocess_df(test_df)

# Target: class_index (ensure integer 1-10)
train_df["class_index"] = pd.to_numeric(train_df["class_index"], errors="coerce").astype("Int64")
test_df["class_index"] = pd.to_numeric(test_df["class_index"], errors="coerce").astype("Int64")

# Drop rows with missing class (if any)
train_df = train_df.dropna(subset=["class_index"])
test_df = test_df.dropna(subset=["class_index"])

y_train = train_df["class_index"].astype(int).values
y_test = test_df["class_index"].astype(int).values
X_train_raw = train_df["text"].values
X_test_raw = test_df["text"].values

print("Train size:", len(y_train), "Test size:", len(y_test))
print("Class distribution (train):\n",pd.Series(y_train).value_counts().sort_index())
print("Sample combined text:", X_train_raw[0][:200], "...")

Train size: 50000 Test size: 5000
Class distribution (train):
 1      3106
2      5545
3      4065
4      4493
5      6541
6      3656
7     12185
8      3518
9      3282
10     3609
Name: count, dtype: int64
Sample combined text: why doesn't an optical mouse work on a glass table? or even on some surfaces? ...


In [ ]:
rt_index())
print("Sample combined text:", X_train_raw[0][:200], "...")

In [15]:
# TF-IDF vectorization with custom tokenizer (stopwords + lemmatization/stemming)
# lowercase=False because tokenize_for_vectorizer already lowercases
vectorizer = TfidfVectorizer(
    tokenizer=tokenize_for_vectorizer,
    lowercase=False,
    max_features=50_000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents="unicode",
)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)
print("Feature matrix shape train:", X_train.shape, "test:", X_test.shape)
print("Sample tokens (first doc):", tokenize_for_vectorizer(X_train_raw[0])[:25])

/Users/tyleryue/Desktop/4NL3/project/project-4nl3/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Feature matrix shape train: (50000, 50000) test: (5000, 50000)
Sample tokens (first doc): ['optical', 'mouse', 'work', 'glass', 'table', 'even', 'surface']


## 2. Baseline models

- **Random**: predict random class index (1–10).
- **Majority**: predict the most frequent class in the training set.
- **Logistic Regression**: linear classifier on TF-IDF features.
- **Random Forest**: ensemble on TF-IDF features.
- **XGBoost**: gradient boosting on TF-IDF features (if installed).

In [16]:
# --- Random Baseline ---
rng = np.random.default_rng(42)
classes = np.unique(y_train)
y_pred_random = rng.choice(classes, size=len(y_test))
acc_random = accuracy_score(y_test, y_pred_random)
print("Random baseline accuracy:", round(acc_random, 4))
print(classification_report(y_test, y_pred_random, zero_division=0))

Random baseline accuracy: 0.1048
              precision    recall  f1-score   support

           1       0.06      0.08      0.06       354
           2       0.11      0.10      0.10       584
           3       0.08      0.10      0.09       385
           4       0.10      0.12      0.11       434
           5       0.10      0.09      0.10       622
           6       0.07      0.11      0.09       330
           7       0.27      0.10      0.15      1202
           8       0.09      0.14      0.11       330
           9       0.07      0.09      0.08       355
          10       0.11      0.14      0.12       404

    accuracy                           0.10      5000
   macro avg       0.11      0.11      0.10      5000
weighted avg       0.13      0.10      0.11      5000



In [17]:
# --- Majority Vote Baseline ---
from collections import Counter
majority_class = Counter(y_train).most_common(1)[0][0]
y_pred_majority = np.full(len(y_test), majority_class)
acc_majority = accuracy_score(y_test, y_pred_majority)
print("Majority baseline accuracy:", round(acc_majority, 4))
print("Predicted class for all:", majority_class)
print(classification_report(y_test, y_pred_majority, zero_division=0))

Majority baseline accuracy: 0.2404
Predicted class for all: 7
              precision    recall  f1-score   support

           1       0.00      0.00      0.00       354
           2       0.00      0.00      0.00       584
           3       0.00      0.00      0.00       385
           4       0.00      0.00      0.00       434
           5       0.00      0.00      0.00       622
           6       0.00      0.00      0.00       330
           7       0.24      1.00      0.39      1202
           8       0.00      0.00      0.00       330
           9       0.00      0.00      0.00       355
          10       0.00      0.00      0.00       404

    accuracy                           0.24      5000
   macro avg       0.02      0.10      0.04      5000
weighted avg       0.06      0.24      0.09      5000



In [18]:
# --- Logistic Regression Baseline ---
lr = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression accuracy:", round(acc_lr, 4))
print(classification_report(y_test, y_pred_lr, zero_division=0))

/Users/tyleryue/Desktop/4NL3/project/project-4nl3/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression accuracy: 0.5998
              precision    recall  f1-score   support

           1       0.58      0.31      0.41       354
           2       0.67      0.65      0.66       584
           3       0.66      0.62      0.64       385
           4       0.51      0.34      0.41       434
           5       0.76      0.79      0.77       622
           6       0.88      0.66      0.75       330
           7       0.44      0.65      0.53      1202
           8       0.67      0.50      0.57       330
           9       0.65      0.64      0.64       355
          10       0.76      0.58      0.65       404

    accuracy                           0.60      5000
   macro avg       0.66      0.57      0.60      5000
weighted avg       0.62      0.60      0.60      5000



In [24]:
# --- Random Forest Baseline ---
rf = RandomForestClassifier(n_estimators=200, max_depth=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest accuracy:", round(acc_rf, 4))
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest accuracy: 0.4706
              precision    recall  f1-score   support

           1       0.76      0.16      0.27       354
           2       0.73      0.25      0.38       584
           3       0.74      0.30      0.43       385
           4       0.63      0.14      0.23       434
           5       0.76      0.68      0.71       622
           6       0.90      0.49      0.64       330
           7       0.32      0.83      0.46      1202
           8       0.70      0.26      0.38       330
           9       0.62      0.49      0.55       355
          10       0.80      0.32      0.46       404

    accuracy                           0.47      5000
   macro avg       0.70      0.39      0.45      5000
weighted avg       0.64      0.47      0.46      5000

